# Unity Catalog Setup for Adventure Works Data Warehouse

This notebook sets up the complete Unity Catalog structure:

## Execution Sequence:
1. **Step 1**: Create the `adwm_wh` catalog with managed location
2. **Step 2**: Create medallion architecture schemas (bronze, silver, gold, utilities)
3. **Step 3**: Create volumes schema and external volume for landing files

## Prerequisites:
- External location `adwm_wh_location` must exist
- Storage credential `adwm_storage_cred` must be configured
- Azure permissions must be granted on storage containers

## Result:
A fully configured Unity Catalog with:
- Catalog: `adwm_wh`
- Schemas: `bronze`, `silver`, `gold`, `utilities`, `volumes`
- External Volume: `adwm_wh.volumes.landing_files`

In [0]:
%sql
-- Create catalog using the manually created external location
CREATE CATALOG IF NOT EXISTS `adwm_wh`
MANAGED LOCATION 'abfss://uc-adwm@adwmstg.dfs.core.windows.net/adwm_wh'
COMMENT 'ETL workspace catalog for Adventure Works data';

In [0]:
%sql
-- Create schemas in the new catalog
CREATE SCHEMA IF NOT EXISTS adwm_wh.bronze COMMENT 'Raw/landing zone data';
CREATE SCHEMA IF NOT EXISTS adwm_wh.silver COMMENT 'Cleansed and conformed data';
CREATE SCHEMA IF NOT EXISTS adwm_wh.gold COMMENT 'Business-level aggregates';
CREATE SCHEMA IF NOT EXISTS adwm_wh.utilities COMMENT 'Helper tables and functions';

In [0]:
%sql
-- Create volumes schema
CREATE SCHEMA IF NOT EXISTS adwm_wh.volumes COMMENT 'Schema for Unity Catalog volumes';

-- Create an EXTERNAL volume in the volumes schema using the external location URL
CREATE EXTERNAL VOLUME IF NOT EXISTS adwm_wh.volumes.landing_files
LOCATION 'abfss://sharif@adwmstg.dfs.core.windows.net/'
COMMENT 'Volume for landing files in separate container';

-- Create volume for storing streaming checkpoints
CREATE VOLUME IF NOT EXISTS adwm_wh.volumes.checkpoints
COMMENT 'Storage for Auto Loader streaming checkpoints';

In [0]:
# Verify the catalog and all schemas were created successfully
print("=== Catalog Information ===")
spark.sql("DESCRIBE CATALOG adwm_wh").display()

print("\n=== Schemas in adwm_wh ===")
spark.sql("SHOW SCHEMAS IN adwm_wh").display()

print("\n=== Volumes in adwm_wh.volumes ===")
spark.sql("SHOW VOLUMES IN adwm_wh.volumes").display()